## 单周期数据通路：把指令位段接到控制线上

> 2026-10-22 · 来源：nand2tetris 项目复习（modules/408/NAND2TETRIS.md）
- CPU 接口：`inM`（= `RAM[A]`）、`instruction`、`reset` → `outM`、`writeM`、`addressM`、`pc`
- **A 寄存器的 load = 「是 A 指令」OR「dest 含 A 位」**；只写前者会导致 C-指令的 `AMD` 组合写不进 A
- `writeM = i[15] AND dest 的 M 位`：A-指令没有 dest 字段（那几位是常量），不屏蔽会误写内存
- ALU 的 `x` 恒接 D 寄存器，`y` 由 `a` 位在 A/M 间选；`addressM` = A 寄存器的输出（不是 PC）
- PC：`inc` 恒 1，`load` = 跳转条件成立，优先级 `reset > load > inc`
- 时序：`outM`/`writeM` 是**组合**的（当周期有效），`addressM`/`pc` 是**时钟**的（下周期生效）
- 这台 CPU **没有微程序、没有状态机**：译码 = 位段直连控制线。代价是完全没有流水线 —— 取指与执行共用一条数据通路，切不开


## 一条指令 = 一组控制信号：iaddq 在 SEQ 里只改两处

> 2026-10-22 · 来源：CSAPP 项目复习（`modules/408/CSAPP.md`） §7.4（实践仓 `learning-408/csapp/archlab/`，本地 PDF 是 Y86-64 + iaddq 的 3e 版本）
`iaddq V, rB`：`R[rB] <- R[rB] + V` **并设置条件码**。它存在的理由：Y86-64 的 `addq` 只能加寄存器、`irmovq` 不能做加法，`rB += 8` 原本要两条指令 —— 这就是 CISC/RISC 之争的缩影（408 必考对比）。

**与 `irmovq` 的差别只有两处**（其余信号全同）：
| 信号 | iaddq | irmovq |
|---|---|---|
| `srcB` | `rB`（要读寄存器） | `RNONE` |
| `set_cc` | `1` | `0` |

其余：`need_regids = 1`、`srcA = RNONE`、`aluA = valB`、`aluB = valC`、`dstE = rB`。

**408 对应**：数据通路与控制信号、指令周期、流水线冒险 —— 考点重要，但 Y86 是玩具 ISA，本 lab 只做 Part A（汇编）+ Part B（iaddq）建立直觉，**Part C 的 CPE 优化是无底洞，放弃**。
